# 01 · Results tables (camera-ready)

Thin viewer over `waveletflux.analysis`. The **source of truth** is

```bash
python scripts/reproduce_tables.py        # writes artifacts/tables/{tex,csv} and artifacts/checks/
python scripts/check_paper_reproduction.py  # proves the builders reproduce the submitted paper
```

| Paper | Builder | Time averaging |
|---|---|---|
| Table 1, S10 | `selection_table` | horizon-balanced |
| S9 = S13 | `bands_table` | horizon-balanced |
| S11, S12 | `horizon_table` | per horizon |
| Table 3, S14, S16, S17 | `param_table` | per step |
| S15, S18 | `spectral_param_table` | per step |
| rebuttal: RB + LatentFM | `rb_summary_table` | horizon-balanced (spectral) |
| rebuttal: kinetic energy | `physics.ke_table` | per horizon |
| rebuttal: IC perturbation | `ic_sensitivity.ic_tables` | per horizon / per step |
| Tables 2, S19, S20 | `profiling.load_profiles` | – |

Every cell = mean ± std over inference seeds 42/84/168 (deterministic baselines: one run).

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from waveletflux.analysis import catalog as C, tables as T
from waveletflux.analysis.physics import ke_table
from waveletflux.analysis.ic_sensitivity import ic_tables, load_ic
from waveletflux.analysis.profiling import load_profiles, selected
pd.set_option("display.max_columns", 40)

df, gdf = C.load()          # per-field rows, _global_ physics rows

loaded 856,572 rows from 111 files (736,308 per-field, 120,264 global)


## Table 1 — wavelet selection (band- and horizon-averaged spectral coherence RMSE)

In [2]:
sel = T.across_seeds(T.selection_table, df, models=C.ROWS_GEN, time_agg="horizon")
print("selection consistent with BEST_WFM_PER_PDE:", not T.check_selection(sel[0]))
display(T.s10_table(sel[0], sel[1]))
T.show(*sel[:2]).rename(columns=C.PDE_LABELS)

selection consistent with BEST_WFM_PER_PDE: True


,pde,scale,wavelet,mean,std
0,turbulent_radiative_layer_2D,1scale,haar,0.147050,0.000208
1,turbulent_radiative_layer_2D,3scale,haar,0.149129,0.000277
2,rayleigh_benard,1scale,db4,0.121084,0.001956
3,rayleigh_benard,3scale,db4,0.109106,0.001077
4,active_matter,1scale,haar,0.241397,0.000665
5,active_matter,3scale,db2,0.219940,0.000322


,Turb. Rad. Layer 2D,Rayleigh--B\'{e}nard,Active Matter
WFM-1 haar,0.147 ± 0.000,0.140 ± 0.001,0.241 ± 0.001
WFM-1 db2,0.154 ± 0.000,0.132 ± 0.001,0.288 ± 0.000
WFM-1 db4,0.149 ± 0.000,0.121 ± 0.002,0.303 ± 0.000
WFM-1 db6,0.150 ± 0.000,0.140 ± 0.002,0.290 ± 0.000
WFM-3 haar,0.149 ± 0.000,0.131 ± 0.001,0.247 ± 0.001
WFM-3 db2,0.154 ± 0.001,0.111 ± 0.001,0.220 ± 0.000
WFM-3 db4,0.154 ± 0.000,0.109 ± 0.001,0.249 ± 0.000
WFM-3 db6,0.153 ± 0.001,0.112 ± 0.000,0.238 ± 0.000
FM$_\text{pixel}$,0.153 ± 0.001,0.131 ± 0.001,0.244 ± 0.001
FM$_\text{fourier}$,0.387 ± 0.001,0.246 ± 0.001,0.324 ± 0.000


## S11 / S12 — VRMSE and CRPS per rollout horizon

In [3]:
T.show(*T.across_seeds(T.horizon_table, df, metric="vrmse", models=C.ROWS_ALL)[:2])

turbulent_radiative_layer_2D                 \
                                           01:32          33:64   
WFM-1 haar                         0.544 ± 0.011  0.791 ± 0.016   
WFM-1 db2                          0.550 ± 0.003  0.750 ± 0.010   
WFM-1 db4                          0.550 ± 0.007  0.730 ± 0.007   
WFM-1 db6                          0.536 ± 0.005  0.703 ± 0.009   
WFM-3 haar                         0.549 ± 0.007  0.824 ± 0.017   
WFM-3 db2                          0.655 ± 0.006  0.893 ± 0.016   
WFM-3 db4                          0.586 ± 0.015  0.759 ± 0.007   
WFM-3 db6                          0.591 ± 0.005  0.879 ± 0.007   
FM$_\text{pixel}$                  0.574 ± 0.002  0.891 ± 0.010   
FM$_\text{fourier}$                0.635 ± 0.010  0.773 ± 0.016   
WDNO                               1.517 ± 0.000  1.834 ± 0.001   
WNO                                          inf            inf   
FNO                                        2.952          4.978   
TFNO                                       2.300          6.534   

                                   rayleigh_benard                 \
                             65:97           01:32          33:64   
WFM-1 haar           0.789 ± 0.014   1.782 ± 0.040  2.020 ± 0.049   
WFM-1 db2            0.850 ± 0.024   1.333 ± 0.046  1.844 ± 0.103   
WFM-1 db4            0.811 ± 0.001   1.689 ± 0.016  2.271 ± 0.077   
WFM-1 db6            0.786 ± 0.017   1.441 ± 0.012  1.949 ± 0.028   
WFM-3 haar           0.797 ± 0.014   1.559 ± 0.032  2.318 ± 0.095   
WFM-3 db2            0.981 ± 0.022   1.051 ± 0.043  1.729 ± 0.048   
WFM-3 db4            0.803 ± 0.006   1.029 ± 0.059  1.869 ± 0.048   
WFM-3 db6            0.831 ± 0.019   0.755 ± 0.025  1.598 ± 0.059   
FM$_\text{pixel}$    0.830 ± 0.016   1.319 ± 0.030  1.962 ± 0.030   
FM$_\text{fourier}$  0.804 ± 0.007   2.331 ± 0.057  1.417 ± 0.056   
WDNO                 1.953 ± 0.003  19.262 ± 0.012  8.171 ± 0.032   
WNO                             --          44.226         24.463   
FNO                          4.349           3.368          3.972   
TFNO                         6.387         102.678         31.842   

                                     active_matter                 \
                            65:196           01:16          17:32   
WFM-1 haar           0.788 ± 0.023   2.847 ± 0.015  1.541 ± 0.011   
WFM-1 db2            0.791 ± 0.010   5.652 ± 0.019  2.930 ± 0.004   
WFM-1 db4            0.802 ± 0.015  14.227 ± 0.027  5.574 ± 0.011   
WFM-1 db6            1.025 ± 0.019   6.732 ± 0.009  2.927 ± 0.012   
WFM-3 haar           0.807 ± 0.017   0.848 ± 0.008  1.327 ± 0.015   
WFM-3 db2            0.766 ± 0.008   0.933 ± 0.011  1.349 ± 0.033   
WFM-3 db4            0.770 ± 0.009   3.242 ± 0.013  2.271 ± 0.021   
WFM-3 db6            0.834 ± 0.018   2.341 ± 0.010  2.118 ± 0.012   
FM$_\text{pixel}$    0.821 ± 0.025   0.580 ± 0.004  1.380 ± 0.018   
FM$_\text{fourier}$  0.870 ± 0.005   1.386 ± 0.003  1.126 ± 0.008   
WDNO                 1.392 ± 0.002   4.475 ± 0.011  2.181 ± 0.002   
WNO                          2.452          10.669          4.321   
FNO                            inf           1.682          3.227   
TFNO                         2.965           4.443          2.968   

                                    
                             33:77  
WFM-1 haar           1.079 ± 0.004  
WFM-1 db2            1.622 ± 0.003  
WFM-1 db4            2.488 ± 0.001  
WFM-1 db6            1.553 ± 0.010  
WFM-3 haar           1.022 ± 0.002  
WFM-3 db2            1.085 ± 0.007  
WFM-3 db4            1.418 ± 0.005  
WFM-3 db6            1.276 ± 0.006  
FM$_\text{pixel}$    1.028 ± 0.002  
FM$_\text{fourier}$  1.007 ± 0.003  
WDNO                 1.400 ± 0.000  
WNO                          2.002  
FNO                          1.727  
TFNO                         1.525

In [4]:
T.show(*T.across_seeds(T.horizon_table, df, metric="crps", models=C.ROWS_GEN)[:2])

turbulent_radiative_layer_2D                 \
                                           01:32          33:64   
WFM-1 haar                         0.492 ± 0.006  1.047 ± 0.033   
WFM-1 db2                          0.447 ± 0.007  1.097 ± 0.008   
WFM-1 db4                          0.521 ± 0.006  1.052 ± 0.002   
WFM-1 db6                          0.451 ± 0.003  0.983 ± 0.017   
WFM-3 haar                         0.410 ± 0.003  1.018 ± 0.011   
WFM-3 db2                          0.465 ± 0.008  1.133 ± 0.006   
WFM-3 db4                          0.427 ± 0.004  0.984 ± 0.007   
WFM-3 db6                          0.451 ± 0.006  1.026 ± 0.006   
FM$_\text{pixel}$                  0.418 ± 0.006  1.088 ± 0.007   
FM$_\text{fourier}$                1.112 ± 0.003  1.632 ± 0.007   
WDNO                               2.460 ± 0.002  3.374 ± 0.004   

                                   rayleigh_benard                 \
                             65:97           01:32          33:64   
WFM-1 haar           1.218 ± 0.032   0.005 ± 0.000  0.037 ± 0.000   
WFM-1 db2            1.316 ± 0.032   0.005 ± 0.000  0.036 ± 0.000   
WFM-1 db4            1.241 ± 0.021   0.007 ± 0.000  0.041 ± 0.000   
WFM-1 db6            1.188 ± 0.008   0.005 ± 0.000  0.040 ± 0.000   
WFM-3 haar           1.178 ± 0.030   0.004 ± 0.000  0.037 ± 0.000   
WFM-3 db2            1.179 ± 0.026   0.003 ± 0.000  0.032 ± 0.000   
WFM-3 db4            1.130 ± 0.016   0.004 ± 0.000  0.032 ± 0.000   
WFM-3 db6            1.200 ± 0.015   0.004 ± 0.000  0.031 ± 0.001   
FM$_\text{pixel}$    1.210 ± 0.017   0.005 ± 0.000  0.036 ± 0.001   
FM$_\text{fourier}$  1.772 ± 0.011   0.013 ± 0.000  0.040 ± 0.001   
WDNO                 4.284 ± 0.013   0.050 ± 0.000  0.102 ± 0.000   

                                    active_matter                 \
                            65:196          01:16          17:32   
WFM-1 haar           0.074 ± 0.003  0.119 ± 0.001  0.190 ± 0.001   
WFM-1 db2            0.073 ± 0.002  0.173 ± 0.000  0.239 ± 0.000   
WFM-1 db4            0.073 ± 0.003  0.326 ± 0.000  0.388 ± 0.000   
WFM-1 db6            0.090 ± 0.001  0.175 ± 0.000  0.222 ± 0.000   
WFM-3 haar           0.077 ± 0.004  0.080 ± 0.000  0.169 ± 0.003   
WFM-3 db2            0.069 ± 0.002  0.087 ± 0.001  0.176 ± 0.003   
WFM-3 db4            0.069 ± 0.001  0.109 ± 0.001  0.209 ± 0.001   
WFM-3 db6            0.079 ± 0.003  0.100 ± 0.001  0.197 ± 0.001   
FM$_\text{pixel}$    0.076 ± 0.005  0.059 ± 0.000  0.166 ± 0.001   
FM$_\text{fourier}$  0.090 ± 0.001  0.197 ± 0.001  0.216 ± 0.002   
WDNO                 0.138 ± 0.000  0.218 ± 0.000  0.245 ± 0.000   

                                    
                             33:77  
WFM-1 haar           0.198 ± 0.001  
WFM-1 db2            0.261 ± 0.000  
WFM-1 db4            0.387 ± 0.000  
WFM-1 db6            0.231 ± 0.000  
WFM-3 haar           0.191 ± 0.001  
WFM-3 db2            0.196 ± 0.002  
WFM-3 db4            0.239 ± 0.000  
WFM-3 db6            0.213 ± 0.001  
FM$_\text{pixel}$    0.194 ± 0.001  
FM$_\text{fourier}$  0.211 ± 0.001  
WDNO                 0.266 ± 0.000

## S13 — spectral coherence RMSE per band

In [5]:
T.show(*T.across_seeds(T.bands_table, df, models=C.ROWS_GEN, time_agg="horizon")[:2])

turbulent_radiative_layer_2D                 \
                                             Low            Mid   
WFM-1 haar                         0.132 ± 0.000  0.175 ± 0.001   
WFM-1 db2                          0.139 ± 0.000  0.181 ± 0.000   
WFM-1 db4                          0.132 ± 0.000  0.175 ± 0.000   
WFM-1 db6                          0.133 ± 0.001  0.175 ± 0.000   
WFM-3 haar                         0.133 ± 0.001  0.174 ± 0.000   
WFM-3 db2                          0.139 ± 0.002  0.188 ± 0.001   
WFM-3 db4                          0.141 ± 0.001  0.178 ± 0.000   
WFM-3 db6                          0.136 ± 0.000  0.175 ± 0.001   
FM$_\text{pixel}$                  0.141 ± 0.001  0.181 ± 0.001   
FM$_\text{fourier}$                0.240 ± 0.002  0.686 ± 0.000   
WDNO                               0.199 ± 0.000  0.249 ± 0.000   

                                   rayleigh_benard                 \
                              High             Low            Mid   
WFM-1 haar           0.134 ± 0.000   0.171 ± 0.002  0.132 ± 0.000   
WFM-1 db2            0.142 ± 0.000   0.167 ± 0.001  0.126 ± 0.001   
WFM-1 db4            0.140 ± 0.000   0.168 ± 0.005  0.118 ± 0.001   
WFM-1 db6            0.141 ± 0.000   0.196 ± 0.003  0.131 ± 0.001   
WFM-3 haar           0.141 ± 0.000   0.149 ± 0.001  0.130 ± 0.001   
WFM-3 db2            0.136 ± 0.000   0.141 ± 0.003  0.113 ± 0.001   
WFM-3 db4            0.144 ± 0.000   0.143 ± 0.002  0.109 ± 0.001   
WFM-3 db6            0.148 ± 0.000   0.147 ± 0.001  0.110 ± 0.000   
FM$_\text{pixel}$    0.138 ± 0.001   0.149 ± 0.001  0.129 ± 0.000   
FM$_\text{fourier}$  0.236 ± 0.000   0.294 ± 0.002  0.332 ± 0.000   
WDNO                 0.366 ± 0.000   0.392 ± 0.001  0.474 ± 0.000   

                                    active_matter                 \
                              High            Low            Mid   
WFM-1 haar           0.116 ± 0.001  0.229 ± 0.002  0.291 ± 0.000   
WFM-1 db2            0.105 ± 0.001  0.291 ± 0.000  0.336 ± 0.000   
WFM-1 db4            0.077 ± 0.000  0.306 ± 0.001  0.351 ± 0.000   
WFM-1 db6            0.093 ± 0.002  0.283 ± 0.001  0.346 ± 0.000   
WFM-3 haar           0.114 ± 0.001  0.205 ± 0.001  0.280 ± 0.001   
WFM-3 db2            0.079 ± 0.001  0.199 ± 0.001  0.262 ± 0.000   
WFM-3 db4            0.075 ± 0.001  0.219 ± 0.001  0.292 ± 0.000   
WFM-3 db6            0.080 ± 0.001  0.196 ± 0.001  0.258 ± 0.000   
FM$_\text{pixel}$    0.115 ± 0.001  0.183 ± 0.002  0.258 ± 0.000   
FM$_\text{fourier}$  0.113 ± 0.000  0.436 ± 0.001  0.534 ± 0.000   
WDNO                 0.535 ± 0.000  0.367 ± 0.001  0.345 ± 0.000   

                                    
                              High  
WFM-1 haar           0.204 ± 0.000  
WFM-1 db2            0.236 ± 0.000  
WFM-1 db4            0.251 ± 0.000  
WFM-1 db6            0.242 ± 0.000  
WFM-3 haar           0.256 ± 0.000  
WFM-3 db2            0.198 ± 0.000  
WFM-3 db4            0.235 ± 0.000  
WFM-3 db6            0.260 ± 0.000  
FM$_\text{pixel}$    0.291 ± 0.000  
FM$_\text{fourier}$  0.001 ± 0.000  
WDNO                 0.200 ± 0.000

## Table 3 / S14 / S16 / S17 — CRPS per physical parameter

In [6]:
for pde, param in [("turbulent_radiative_layer_2D", "tcool"), ("rayleigh_benard", "Rayleigh"), ("rayleigh_benard", "Prandtl")]:
    m, s, _ = T.across_seeds(T.param_table, df, pde=pde, param=param, models=C.ROWS_GEN, time_agg="step")
    m.columns = s.columns = [C.PARAM_LABELS[param](c) for c in m.columns]
    print(pde, param); display(T.show(m, s))

turbulent_radiative_layer_2D tcool


,0.03,0.06,0.1,0.18,0.32,0.56,1,1.78,3.16
WFM-1 haar,1.165 ± 0.043,1.135 ± 0.057,0.865 ± 0.052,1.044 ± 0.028,0.830 ± 0.051,0.798 ± 0.030,0.806 ± 0.056,0.983 ± 0.036,0.673 ± 0.023
WFM-1 db2,1.112 ± 0.031,1.078 ± 0.025,0.878 ± 0.042,1.057 ± 0.040,0.880 ± 0.065,0.779 ± 0.015,0.853 ± 0.053,0.976 ± 0.051,1.002 ± 0.061
WFM-1 db4,1.170 ± 0.048,1.102 ± 0.044,0.847 ± 0.043,1.084 ± 0.052,0.875 ± 0.005,0.849 ± 0.019,0.879 ± 0.065,0.946 ± 0.038,0.719 ± 0.036
WFM-1 db6,1.076 ± 0.023,1.074 ± 0.014,0.831 ± 0.021,1.041 ± 0.017,0.823 ± 0.021,0.773 ± 0.034,0.741 ± 0.016,0.943 ± 0.018,0.594 ± 0.040
WFM-3 haar,1.107 ± 0.012,1.117 ± 0.034,0.868 ± 0.019,0.988 ± 0.067,0.773 ± 0.020,0.765 ± 0.046,0.675 ± 0.026,0.961 ± 0.011,0.595 ± 0.024
WFM-3 db2,1.248 ± 0.024,1.141 ± 0.044,0.894 ± 0.043,1.027 ± 0.025,0.836 ± 0.017,0.783 ± 0.041,0.751 ± 0.040,1.019 ± 0.008,0.656 ± 0.028
WFM-3 db4,1.048 ± 0.019,1.006 ± 0.025,0.817 ± 0.042,0.982 ± 0.049,0.817 ± 0.033,0.749 ± 0.027,0.692 ± 0.028,0.906 ± 0.005,0.632 ± 0.011
WFM-3 db6,1.075 ± 0.018,1.034 ± 0.015,0.882 ± 0.038,1.000 ± 0.053,0.863 ± 0.039,0.850 ± 0.027,0.770 ± 0.048,0.969 ± 0.038,0.617 ± 0.006
FM$_\text{pixel}$,1.220 ± 0.040,1.126 ± 0.016,0.875 ± 0.045,1.061 ± 0.048,0.855 ± 0.046,0.783 ± 0.004,0.731 ± 0.012,0.978 ± 0.021,0.551 ± 0.030
FM$_\text{fourier}$,1.709 ± 0.031,1.667 ± 0.032,1.424 ± 0.006,1.618 ± 0.012,1.485 ± 0.019,1.430 ± 0.016,1.451 ± 0.014,1.595 ± 0.015,1.194 ± 0.003


rayleigh_benard Rayleigh


,$10^{6}$,$10^{8}$,$10^{10}$
WFM-1 haar,0.059 ± 0.001,0.060 ± 0.005,0.051 ± 0.001
WFM-1 db2,0.052 ± 0.002,0.062 ± 0.003,0.052 ± 0.003
WFM-1 db4,0.059 ± 0.008,0.060 ± 0.003,0.051 ± 0.001
WFM-1 db6,0.060 ± 0.004,0.069 ± 0.004,0.075 ± 0.001
WFM-3 haar,0.060 ± 0.004,0.060 ± 0.003,0.055 ± 0.003
WFM-3 db2,0.054 ± 0.004,0.055 ± 0.001,0.049 ± 0.001
WFM-3 db4,0.053 ± 0.005,0.058 ± 0.002,0.048 ± 0.003
WFM-3 db6,0.065 ± 0.006,0.055 ± 0.002,0.057 ± 0.001
FM$_\text{pixel}$,0.057 ± 0.007,0.066 ± 0.004,0.051 ± 0.002
FM$_\text{fourier}$,0.069 ± 0.001,0.076 ± 0.001,0.062 ± 0.001


rayleigh_benard Prandtl


,$10^{-1}$,$1$,$10^{1}$
WFM-1 haar,0.091 ± 0.001,0.050 ± 0.004,0.029 ± 0.001
WFM-1 db2,0.088 ± 0.005,0.049 ± 0.002,0.030 ± 0.001
WFM-1 db4,0.088 ± 0.007,0.048 ± 0.002,0.034 ± 0.001
WFM-1 db6,0.087 ± 0.003,0.066 ± 0.004,0.051 ± 0.001
WFM-3 haar,0.092 ± 0.007,0.050 ± 0.002,0.033 ± 0.000
WFM-3 db2,0.079 ± 0.003,0.048 ± 0.001,0.031 ± 0.001
WFM-3 db4,0.079 ± 0.005,0.051 ± 0.004,0.028 ± 0.001
WFM-3 db6,0.086 ± 0.007,0.050 ± 0.002,0.040 ± 0.001
FM$_\text{pixel}$,0.088 ± 0.008,0.054 ± 0.001,0.033 ± 0.001
FM$_\text{fourier}$,0.111 ± 0.001,0.058 ± 0.001,0.038 ± 0.001


## Rebuttal tables — RB with LatentFM, kinetic energy, IC perturbation

In [7]:
rows = C.ROWS_BEST_GEN + [C.LATENTFM]
T.show(*T.across_seeds(T.rb_summary_table, T.best_view(df, keep=C.GEN_MODELS + [C.LATENTFM]), models=rows)[:2])

Spectral coherence RMSE                                \
                                         Low            Mid           High   
WFM 1-scale                    0.168 ± 0.005  0.118 ± 0.001  0.077 ± 0.000   
WFM 3-scale                    0.143 ± 0.002  0.109 ± 0.001  0.075 ± 0.001   
FM$_\text{pixel}$              0.149 ± 0.001  0.129 ± 0.000  0.115 ± 0.001   
FM$_\text{fourier}$            0.294 ± 0.002  0.332 ± 0.000  0.113 ± 0.000   
WDNO                           0.392 ± 0.001  0.474 ± 0.000  0.535 ± 0.000   
LatentFM$_{\div 64}$           0.196 ± 0.000  0.159 ± 0.000  0.131 ± 0.001   

                               CRPS                                
                              01:32          33:64         65:196  
WFM 1-scale           0.007 ± 0.000  0.041 ± 0.000  0.073 ± 0.003  
WFM 3-scale           0.004 ± 0.000  0.032 ± 0.000  0.069 ± 0.001  
FM$_\text{pixel}$     0.005 ± 0.000  0.036 ± 0.001  0.076 ± 0.005  
FM$_\text{fourier}$   0.013 ± 0.000  0.040 ± 0.001  0.090 ± 0.001  
WDNO                  0.050 ± 0.000  0.102 ± 0.000  0.138 ± 0.000  
LatentFM$_{\div 64}$  0.026 ± 0.000  0.057 ± 0.000  0.088 ± 0.004

In [8]:
T.show(*ke_table(T.best_view(gdf, keep=C.GEN_MODELS), C.ROWS_BEST_GEN)[:2])

turbulent_radiative_layer_2D                 \
                                           01:32          33:64   
WFM 1-scale                        0.047 ± 0.001  0.113 ± 0.002   
WFM 3-scale                        0.017 ± 0.001  0.067 ± 0.003   
FM$_\text{pixel}$                  0.016 ± 0.001  0.067 ± 0.004   
FM$_\text{fourier}$                0.054 ± 0.001  0.096 ± 0.006   
WDNO                               0.356 ± 0.000  0.225 ± 0.001   

                                   rayleigh_benard                 \
                             65:97           01:32          33:64   
WFM 1-scale          0.221 ± 0.006   0.007 ± 0.000  0.410 ± 0.005   
WFM 3-scale          0.147 ± 0.004   0.003 ± 0.000  0.217 ± 0.006   
FM$_\text{pixel}$    0.177 ± 0.007   0.004 ± 0.000  0.235 ± 0.005   
FM$_\text{fourier}$  0.169 ± 0.011   0.015 ± 0.000  0.157 ± 0.002   
WDNO                 0.457 ± 0.002   1.014 ± 0.002  2.506 ± 0.007   

                                    active_matter                 \
                            65:196          01:16          17:32   
WFM 1-scale          0.379 ± 0.010  0.502 ± 0.001  0.538 ± 0.005   
WFM 3-scale          0.348 ± 0.023  0.112 ± 0.002  0.391 ± 0.011   
FM$_\text{pixel}$    0.251 ± 0.010  0.051 ± 0.001  0.288 ± 0.012   
FM$_\text{fourier}$  0.919 ± 0.006  0.503 ± 0.002  0.538 ± 0.013   
WDNO                 2.309 ± 0.003  0.747 ± 0.002  0.890 ± 0.002   

                                    
                             33:77  
WFM 1-scale          0.508 ± 0.001  
WFM 3-scale          0.531 ± 0.003  
FM$_\text{pixel}$    0.320 ± 0.010  
FM$_\text{fourier}$  0.692 ± 0.008  
WDNO                 1.052 ± 0.002

In [9]:
ic = ic_tables(load_ic())
display(T.show(*ic["crps"]).rename(index=lambda e: f"eta={e:g}"))
T.show(*ic["spectral"]).rename(index=lambda e: f"eta={e:g}")

,01:32,33:64,65:97
ic_noise,,,
eta=0,0.410 ± 0.003,1.018 ± 0.011,1.178 ± 0.030
eta=0.001,0.410 ± 0.003,1.036 ± 0.016,1.180 ± 0.024
eta=0.01,0.410 ± 0.003,1.030 ± 0.016,1.190 ± 0.007
eta=0.1,0.413 ± 0.003,1.026 ± 0.040,1.203 ± 0.012


,Low,Mid,High
ic_noise,,,
eta=0,0.133 ± 0.001,0.174 ± 0.000,0.141 ± 0.000
eta=0.001,0.133 ± 0.001,0.174 ± 0.001,0.141 ± 0.000
eta=0.01,0.133 ± 0.001,0.174 ± 0.001,0.141 ± 0.001
eta=0.1,0.133 ± 0.000,0.174 ± 0.000,0.141 ± 0.001


## Tables 2 / S19 — inference profiling

In [10]:
selected(load_profiles()).round(3)

,pde,model,wall_clock_s,sec_per_step,gpu_mem_GiB,throughput_Mfr_s,speedup,mem_ratio
0,turbulent_radiative_layer_2D,FM_data_space_v4.0,247.75,0.035,0.590,1.386,1.000,1.000
5,turbulent_radiative_layer_2D,WaveletFlow_1scale_haar_v4.0,198.06,0.028,0.591,1.733,1.251,0.998
9,turbulent_radiative_layer_2D,WaveletFlow_3scales_haar_v4.0,218.27,0.031,0.663,1.573,1.135,0.890
10,rayleigh_benard,FM_data_space_v4.0,633.85,0.045,1.441,1.459,1.000,1.000
13,rayleigh_benard,WaveletFlow_1scale_db4_v4.0,418.13,0.030,1.444,2.212,1.516,0.998
17,rayleigh_benard,WaveletFlow_3scales_db4_v4.0,457.89,0.032,1.515,2.020,1.384,0.951
20,active_matter,FM_data_space_v4.0,310.77,0.046,1.629,1.429,1.000,1.000
25,active_matter,WaveletFlow_1scale_haar_v4.0,213.28,0.031,1.636,2.082,1.457,0.996
26,active_matter,WaveletFlow_3scales_db2_v4.0,247.36,0.037,1.713,1.795,1.256,0.951


## Seed spread audit

In [11]:
pd.read_csv(PROJECT_ROOT / "artifacts/checks/seed_spread_summary.csv")

,table,cells,median,max,pct_over_0p5,pct_over_1
0,R1_rb_latentfm,36,0.66,6.16,58.33,30.56
1,R2_ke_mae,45,1.91,6.53,73.33,64.44
2,S11_vrmse,99,0.99,5.70,71.72,49.49
3,S12_crps,99,0.81,6.16,63.64,41.41
4,S13_spectral_bands,99,0.23,2.89,34.34,13.13
5,S14_crps_tcool,99,2.76,7.43,87.88,81.82
6,S15_spectral_tcool,297,0.83,4.76,68.69,43.10
7,S16_crps_Ra,33,4.50,13.06,90.91,87.88
8,S17_crps_Pr,33,3.17,9.18,87.88,81.82
9,S18a_spectral_Ra,99,1.17,13.16,70.71,53.54
